# <center>**IROS**<center>

**Libraries**

In [1]:
import numpy as np
import mbloodmoon.iros_management as iros
import mbloodmoon as bm

**IROS**

In [2]:
root_path = "/mnt/d/PhD_AASS/Coding/Images_fits/"
mask_file = root_path + "wfm_mask.fits"
simul_data = root_path + "iros_simulation_GC_LMC/20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb/"

cam_a = "cam1a"
cam_b = "cam1b"
dataset = "reconstructed"

filepaths = bm.simulation_files(simul_data)
wfm = bm.codedmask(mask_file, upscale_x=5, upscale_y=1)
sdlA = bm.simulation(filepaths[cam_a][dataset])
sdlB = bm.simulation(filepaths[cam_b][dataset])

max_iterations = 25
snr_threshold = 5

n_test = 0

In [ ]:
iros_output_name: str = root_path + f"iros_output{n_test}.fits"
names = tuple(root_path + f"skyres_IROS_{cam}{n_test}.fits" for cam in (cam_a, cam_b))
comp_name = root_path + f"composed_skyres_IROS_{cam_a.upper()}{cam_b.upper()}{n_test}.fits"

try:
    iros_output = iros.load_iros_output(iros_output_name)
    #resA, snrA = iros.load_sky(names[0])
    #resB, snrB = iros.load_sky(names[1])
    #comp_res, comp_snr = iros.load_sky(comp_name)
    
except FileNotFoundError:
    from mbloodmoon.images import upscale, compose
    iros_output, residuals = iros.perform_iros(
        camerasID=(cam_a, cam_b),
        camera=wfm,
        sdl_camA=sdlA,
        sdl_camB=sdlB,
        max_iterations=max_iterations,
        snr_threshold=snr_threshold,
        dataset=dataset,
    )
    
    iros.save_iros_output(iros_output, mask_file, iros_output_name)

    sdls = (sdlA, sdlB)
    detectors = tuple(bm.count(wfm, sdl.data)[0] for sdl in sdls)
    variances = tuple(bm.variance(wfm, d) for d in detectors)
    snrs = tuple(bm.snratio(sky, np.clip(var_, a_min=1, a_max=None)) for sky, var_ in zip(residuals, variances))

    ups_skies = tuple(upscale(sky, upscale_y=8) for sky in residuals)
    ups_snrs = tuple(upscale(snr, upscale_y=8) for snr in snrs)

    for res, snr, sdl, name in zip(ups_skies, ups_snrs, sdls, names):
        iros.save_sky(res, snr, sdl, name)
    
    iros.save_sky(
        sky=compose(*ups_skies, strict=False)[0],
        snr=compose(*ups_snrs, strict=False)[0],
        sdl=sdlA,
        save_to=comp_name,
    )

# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!


**Compute Parameters**

In [4]:
iros_data_name = root_path + f"iros_data{n_test}.fits"

try:
    iros_data = iros.load_iros_data(iros_data_name)

except FileNotFoundError:
    log = iros.gen_params_log((cam_a, cam_b))

    iros_data = iros.compute_params(
        iros_output=iros_output,
        camera=wfm,
        sdl_camA=sdlA,
        sdl_camB=sdlB,
        log=log,
    )

    # WARNING: the px position in this DB might not match with upscaled skies
    iros.save_iros_data(
        data=iros_data,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=iros_data_name,
    )

# Loading data...
# Loading completed!


**Catalog Comparison**

In [6]:
dataset_name = root_path + f"database{n_test}.fits"

try:
    dataset = iros.load_iros_data(dataset_name)

except FileNotFoundError:
    catalogA = filepaths[cam_a]["sources"]
    catalogB = filepaths[cam_b]["sources"]
    
    # WARNING: source assignment relies only on catalog sources
    dataset = iros.compare_w_catalog(
        data=iros_data,
        catalogA=catalogA,
        catalogB=catalogB,
        camerasID=(cam_a, cam_b),
        min_flux=0.1,
    )

    iros.save_iros_data(
        data=dataset,
        mask_file=mask_file,
        sdls=(sdlA, sdlB),
        save_to=dataset_name,
    )

# Loading data...
# Loading completed!


**Plot for Skies and SNRs**